In [73]:
# Rode esta célula apenas uma vez para instalar tudo na sua venv atual
!pip install langchain==1.4.7 langchain-core==1.7.1 pydantic==2.13.5 langchain-ollama==1.1.0 langgraph==1.2.18 ollama==0.6.3



[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_ollama import ChatOllama

modelo = ChatOllama(
    model='qwen3.5:9b',
    base_url='http://localhost:11434',
    temperature=0.5,
    reasoning=False,
    keep_alive='30m',   
    num_ctx=8192,       
    num_predict=1024,  
)

print('Modelo configurado!')


Modelo configurado!


In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
import re
import time

# ==========================================
# AGENTE 1: ESCRITOR (Montar o texto)
# ==========================================

@tool
def Citation_Formatter(author: str, year: str, page: str = None) -> str:
    """Recebe os dados da fonte e formata em citação ABNT para o corpo do texto."""
    autor_fmt = author.strip().upper()
    ano_fmt = year.strip()
    if page and page.strip():
        return f"({autor_fmt}, {ano_fmt}, p. {page.strip()})"
    return f"({autor_fmt}, {ano_fmt})"

def _formatar_sobrenome_autor(nome_completo: str) -> str:
    partes = nome_completo.strip().split()
    if not partes:
        return ""
    if len(partes) == 1:
        return partes[0].upper()
    return f"{partes[-1].upper()}, {' '.join(partes[:-1])}"

@tool
def Reference_Generator(tipo_fonte: str, dados: dict) -> str:
    """Gera referências bibliográficas de fim de documento (NBR 6023) para artigo, livro ou web."""
    autores = dados.get("autores", [])
    if not autores:
        autores_fmt = "AUTOR DESCONHECIDO"
    elif isinstance(autores, list):
        if len(autores) <= 3:
            autores_fmt = "; ".join([_formatar_sobrenome_autor(a) for a in autores])
        else:
            autores_fmt = f"{_formatar_sobrenome_autor(autores[0])} et al"
    else:
        autores_fmt = _formatar_sobrenome_autor(str(autores))

    titulo = str(dados.get("titulo", "")).strip()
    ano = str(dados.get("ano", "")).strip()

    if tipo_fonte == "artigo":
        return f"{autores_fmt}. {titulo}. **{dados.get('revista', '').strip()}**, v. {dados.get('volume', '').strip()}, n. {dados.get('numero', '').strip()}, p. {dados.get('paginas', '').strip()}, {ano}."
    elif tipo_fonte == "livro":
        edicao = f"{dados.get('edicao', '').strip()}. ed. " if dados.get('edicao') else ""
        return f"{autores_fmt}. **{titulo}**. {edicao}{dados.get('local', '').strip()}: {dados.get('editora', '').strip()}, {ano}."
    elif tipo_fonte == "web":
        return f"{autores_fmt}. **{titulo}**. {ano}. Disponível em: <{dados.get('url', '').strip()}>. Acesso em: {dados.get('data_acesso', '').strip()}."

    return f"{autores_fmt}. {titulo}. {ano}."


# ==========================================
# VALIDAÇÃO ESTRUTURAL (Python puro, sem LLM)
# ==========================================

def validar_estrutura(texto_artigo: str) -> list[str]:
    """Devolve a lista de falhas estruturais (vazia = estrutura OK)."""
    texto_lower = texto_artigo.lower()
    erros = []

    if "introdução" not in texto_lower and "introducao" not in texto_lower:
        erros.append("Seção de 'Introdução' ausente")
    if not any(x in texto_lower for x in ["conclusão", "conclusao", "considerações finais"]):
        erros.append("Seção de 'Conclusão/Considerações Finais' ausente")

    titulos = [l for l in texto_artigo.split('\n') if l.strip().startswith('#')]
    if len(titulos) < 3:
        erros.append("Faltam seções intermediárias de Desenvolvimento (use títulos com #)")

    padrao_parenteses = r"\([A-ZÀ-Úa-zà-ú\s]+,\s?\d{4}[^)]*\)"
    padrao_narrativa = r"[A-ZÀ-Ú][a-zà-ú]+\s\(\d{4}\)"
    if not (re.search(padrao_parenteses, texto_artigo) or re.search(padrao_narrativa, texto_artigo)):
        erros.append("Nenhuma citação no formato ABNT encontrada")

    return erros


In [76]:
writer = create_agent(modelo, tools=[Citation_Formatter, Reference_Generator], system_prompt=(
    'Você é o Escritor. Monte o texto do artigo utilizando formatação ABNT e as ferramentas de citação. '
    'Chame todas as ferramentas de que precisar de uma só vez, na mesma resposta. '
    'Use títulos markdown (#) para Introdução, Desenvolvimento e Conclusão, e inclua a lista de referências ao final. '
    'Em até 20 linhas. Se receber um rascunho e feedback, ajuste o rascunho em vez de começar do zero.'
))

# Checador e Revisor só leem e opinam: sem ferramentas, cada um faz uma única chamada ao modelo.
sources_checker = create_agent(modelo, tools=[], system_prompt=(
    'Você é o Checador de Fontes. Confira se as citações do rascunho parecem existir e fazem sentido no contexto. '
    'Aponte apenas problemas concretos. Responda em até 5 linhas.'
))

revisor = create_agent(modelo, tools=[], system_prompt=(
    'Você é o Revisor. Decida sobre a aprovação do artigo com base no rascunho e no relatório do Checador. '
    'A primeira linha da resposta deve ser exatamente APROVADO ou REPROVADO. '
    'Se reprovar, liste em seguida até 5 ajustes objetivos para o Escritor.'
))


In [ ]:
#LOOP

def veredicto(texto: str) -> str:
    """Primeira palavra da resposta, sem markdown (ex.: '**APROVADO**' -> 'APROVADO')."""
    limpo = re.sub(r'^[^A-Za-z]+', '', texto)
    primeira = limpo.split(maxsplit=1)[0] if limpo else ''
    return re.sub(r'[^A-Z]', '', primeira.upper())

def chamar(agente, conteudo: str, nome: str) -> str:
    inicio = time.perf_counter()
    resposta = agente.invoke(
        {'messages': [{'role': 'user', 'content': conteudo}]},
        config={'recursion_limit': 8}
    )
    texto = resposta['messages'][-1].content
    print(f"\n[{nome}] ({time.perf_counter() - inicio:.1f}s):\n{texto}")
    return texto

contexto = input("qual tema voce quer abordar?")
artigo_bruto = ""
feedback_revisor = ""
artigo_aprovado = False

inicio_total = time.perf_counter()

while not artigo_aprovado:
    tentativas += 1
    print(f"\n--- [Rodada {tentativas}] Iniciando ciclo de produção/ajuste ---")

    # 1. Escritor monta ou ajusta o texto (recebe o rascunho anterior para não recomeçar do zero)
    conteudo_escritor = f"Tema: {contexto}"
    if feedback_revisor:
        conteudo_escritor += (
            f"\n\n[RASCUNHO ANTERIOR]:\n{artigo_bruto}"
            f"\n\n[FEEDBACK DO REVISOR PARA AJUSTAR]:\n{feedback_revisor}"
        )
    artigo_bruto = chamar(writer, conteudo_escritor, "Escritor")

    # 2. Validação estrutural em Python: se falhar, nem chama Checador e Revisor
    erros = validar_estrutura(artigo_bruto)
    if erros:
        feedback_revisor = "Falhas estruturais: " + "; ".join(erros)
        print(f"\n[Validação] REPROVADO sem chamar o LLM. {feedback_revisor}")
        continue

    # 3. Checador analisa as fontes do rascunho atual
    texto_fontes_verificadas = chamar(
        sources_checker, f"Tema: {contexto}\nrascunho:\n{artigo_bruto}", "Checador"
    )

    # 4. Revisor avalia o conjunto completo
    veredicto_revisor = chamar(revisor, (
        f"Tema: {contexto}\n"
        f"rascunho:\n{artigo_bruto}\n"
        f"verificação das fontes:\n{texto_fontes_verificadas}"
    ), "Revisor")

    # 5. Critério de parada
    if veredicto(veredicto_revisor) == "APROVADO":
        artigo_aprovado = True
        print("\n=== PROCESSO CONCLUÍDO: Artigo final aprovado pelo Revisor! ===")
    else:
        feedback_revisor = veredicto_revisor
        print("\n--- Artigo reprovado nesta rodada. Enviando ajustes para o Escritor... ---")

if not artigo_aprovado:
    print(f"\n=== PROCESSO INTERROMPIDO: Limite de {max_tentativas} tentativas atingido sem aprovação final. ===")
print(f"Tempo total: {time.perf_counter() - inicio_total:.1f}s em {tentativas} rodada(s)")



--- [Rodada 1] Iniciando ciclo de produção/ajuste ---

[Escritor] (57.4s):
# Introdução

Os aviões representam uma das maiores conquências da engenharia humana, permitindo a travessia rápida de grandes distâncias. O funcionamento básico envolve a geração de sustentação nas asas e propulsão dos motores para superar a gravidade e o arrasto do ar.

# Desenvolvimento

O princípio fundamental é o teorema de Bernoulli e as leis de Newton: o ar que passa por cima da asa, em formato aerodinâmico (perfil alar), se desloca mais rápido, criando menor pressão superior e gerando sustentação. Simultaneamente, os motores a jato ou hélice empurram o avião para frente, vencendo a resistência do ar. A estabilidade é mantida pelo equilíbrio entre as forças de gravidade, sustentação, arrasto e empuxo, controladas pelos lemes, ailerons e profundores que ajustam o ângulo de ataque e a atitude da aeronave durante o voo.

# Conclusão

Em suma, a operação de um avião é uma complexa interação de princípios fís